In [0]:
SOURCE_PATH = "file:/Workspace/Users/marcos.takayas@atlanteam.com.br/sources"
from pyspark.sql.functions import col
import pandas as pd

In [0]:
df = spark.read \
    .option("inferSchema", "false") \
    .json(f"{SOURCE_PATH}/atendimento_ocorrencias.ndjson")

df.show(truncate=False)
df.printSchema()
df.write.format('delta').mode('overwrite').saveAsTable('bronze.tb_atendimentos')

In [0]:
%pip install openpyxl

In [0]:
df_entrega = (
        spark.read
        .option("multiline", "true")
        .json(f"{SOURCE_PATH}/logistica_entregas.json")
     )
    
df_entregas = df_entrega.select(
    col("delivery_id"),
    col("order_ref"),
    col("carrier.name").alias("carrier_name"),
    col("carrier.mode").alias("carrier_mode"),
    col("delivery_status"),
    col("timestamps.shipped_at").alias("shipped_at"),
    col("timestamps.delivered_at").alias("delivered_at"),
    col("destination.state").alias("state"),
    col("destination.city").alias("city"),
    col("cost"))
    
df_entregas.show(5, truncate=False)
df_entregas.write.format('delta').mode('overwrite').saveAsTable('bronze.tb_entrega')


In [0]:
pdf = pd.read_excel(f"{SOURCE_PATH}/crm_clientes_export.xlsx", engine="openpyxl")
df_clientes = spark.createDataFrame(pdf)
df_clientes.write.format('delta').mode('overwrite').saveAsTable('bronze.dim_clientes')


In [0]:

pdf = pd.read_excel(f"{SOURCE_PATH}/comercial_canais.xlsx", engine="openpyxl")
df_canais = spark.createDataFrame(pdf)
df_canais.show(5, truncate=False)
df_canais.write.format('delta').mode('overwrite').saveAsTable('bronze.dim_canais')


In [0]:
df_produto = (
        spark.read
        .option("multiline", "true")
        .json(f"{SOURCE_PATH}/cadastro_produtos_api_dump.json")
     )
    
df_produtos = df_produto.select(
        col("product.product_id"),
        col("product.name"),
        col("product.category"),
        col("product.subcategory"),
        col("product.status"),
        col("pricing.list_price"),
        col("pricing.currency"),
        col("attributes.family"),
        col("attributes.tags"),
        col("updated_at"))
    
df_produtos.show(5, truncate=False)
df_produtos.write.format('delta').mode('overwrite').saveAsTable('bronze.dim_produto')


In [0]:
df_pedidos_itens = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("sep", ",")
            .option("encoding", "UTF-8")
            .csv(f"{SOURCE_PATH}/erp_pedidos_itens_2025.csv")
    )
df_pedidos_itens.show(5, truncate=False)
df_pedidos_itens.write.format('delta').mode('overwrite').saveAsTable('bronze.tb_pedidos_itens')


In [0]:
df_pedidos_cab = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("sep", ";")
            .option("encoding", "UTF-8")
            .csv(f"{SOURCE_PATH}/erp_pedidos_cabecalho_2025.csv")
    )
df_pedidos_cab.show(5, truncate=False)
df_pedidos_cab.write.format('delta').mode('overwrite').saveAsTable('bronze.tb_pedidos_cabecalho')



In [0]:
df_vendedores = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .csv(f"{SOURCE_PATH}/vendedores.csv")

df_vendedores.write.format('delta').mode('overwrite').saveAsTable('bronze.dim_vendedores')


In [0]:
df_regioes = spark.read \
    .option("header", "true") \
    .option("sep", "|") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .csv(f"{SOURCE_PATH}/legado_regioes_pipe.txt")

df_regioes.write.format('delta').mode('overwrite').saveAsTable('bronze.dim_regioes')
